# Reinforcement Learning (RL): Theory, Math, and Implementation

Reinforcement Learning is the closest thing in Artificial Intelligence to how biological brains actually learn. There is no static dataset. Instead, an **Agent** interacts with an **Environment**. 

Through trial and error, the agent observes the **State** of the world, takes an **Action**, and receives a **Reward** (or punishment). Over time, the agent learns a strategy (a **Policy**) to maximize its total long-term reward.

[Image of Reinforcement learning agent environment loop diagram]

---
### Setup & Imports
Because true RL environments (like Atari games or 3D physics engines) require massive external libraries and heavy GPU processing, we will build custom, lightweight NumPy simulations to visualize exactly how these algorithms learn under the hood!

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, IntSlider, FloatSlider
import scipy.stats as stats

plt.style.use('seaborn-v0_8-darkgrid')

### 1. Q-Learning (Tabular RL)

Q-Learning is a "Value-Based" algorithm. It creates a giant cheat sheet called a **Q-Table**. 
The table has a row for every possible State in the world, and a column for every possible Action. The numbers inside the table are **Q-Values** (Quality Values), which represent the expected long-term reward of taking that specific action in that specific state.

The agent's strategy is extremely simple: "Look at my current state in the table, and pick the action with the highest number."

**Mathematical Foundation (The Bellman Equation):**
How does the table update? After every single step, the agent updates the Q-value of the action it just took by blending its old knowledge with the new reward and the *best possible future* it sees from its new state:
$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ R + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$
* $\alpha$ (Alpha): Learning Rate. How much do we overwrite old knowledge?
* $R$: Immediate reward just received.
* $\gamma$ (Gamma): Discount Factor. How much do we care about long-term future rewards versus immediate rewards?
* $\max_{a'} Q(s', a')$: The highest Q-value available in the *next* state.

**Example Problem:**
* **Simple Gaming:** Navigating a simple maze or playing Tic-Tac-Toe, where the number of possible states is small enough to fit in a spreadsheet.

The interactive visualization below trains a Q-Learning agent to navigate a "Cliff Walking" environment. It must reach the green goal while avoiding the red cliff. Watch how its Q-Table (its brain) evolves over time!

In [ ]:
@interact(episodes=IntSlider(min=0, max=500, step=50, value=0, description='Training Episodes'))
def plot_cliff_walking(episodes):
    rows, cols = 4, 6
    q_table = np.zeros((rows, cols, 4))
    actions = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    
    start, goal = (3, 0), (3, 5)
    cliff = [(3, 1), (3, 2), (3, 3), (3, 4)]
    
    alpha, gamma, epsilon = 0.5, 0.9, 0.1
    np.random.seed(42)
    
    for _ in range(episodes):
        state = start
        while state != goal:
            r, c = state
            
            if np.random.rand() < epsilon:
                a = np.random.randint(4)
            else:
                a = np.argmax(q_table[r, c])
                
            move = actions[a]
            next_state = (max(0, min(rows-1, r + move[0])), max(0, min(cols-1, c + move[1])))
            
            # Environment Rewards
            if next_state in cliff:
                reward = -100
                next_state = start 
                done = True
            elif next_state == goal:
                reward = 10
                done = True
            else:
                reward = -1 
                done = False

            best_next_q = np.max(q_table[next_state[0], next_state[1]])
            q_table[r, c, a] += alpha * (reward + gamma * best_next_q - q_table[r, c, a])
            
            state = next_state
            if done: break

    v_table = np.max(q_table, axis=2)
    v_table[goal] = 10
    for c_pos in cliff: v_table[c_pos] = -100
    
    plt.figure(figsize=(8, 4))
    ax = sns.heatmap(v_table, annot=True, fmt=".0f", cmap='RdYlGn', cbar=False, linewidths=2, linecolor='black')
 
    if episodes > 0:
        for r in range(rows):
            for c in range(cols):
                if (r, c) == goal or (r, c) in cliff: continue
                best_a = np.argmax(q_table[r, c])
                dy, dx = actions[best_a]
                ax.arrow(c + 0.5 - dx*0.2, r + 0.5 - dy*0.2, dx*0.3, dy*0.3, head_width=0.1, fc='black')

    ax.text(start[1]+0.5, start[0]+0.5, 'Start', fontsize=24, ha='center', va='center')
    ax.text(goal[1]+0.5, goal[0]+0.5, 'Finish', fontsize=24, ha='center', va='center')
    for cr, cc in cliff: ax.text(cc+0.5, cr+0.5, 'Cliff', fontsize=24, ha='center', va='center')

    plt.title(f"Q-Learning Cliff Walker (Episodes: {episodes})\nArrows show the learned Policy!", fontsize=14, pad=10)
    plt.axis('off')
    plt.show()

interactive(children=(IntSlider(value=0, description='Training Episodes', max=500, step=50), Output()), _dom_c…

### 2. Deep Q-Networks (DQN)

Q-Learning is great, but it has a fatal flaw: **State Space Explosion**. 
If you want an AI to play Super Mario, the "State" is the raw pixels on the screen. There are more possible pixel combinations on a screen than there are atoms in the universe. You cannot build a spreadsheet that big!

**DQN replaces the Q-Table with a Deep Neural Network.** Instead of looking up a row in a table, the agent feeds the current state (like the pixels of a video game frame) into a Neural Network. The Neural Network outputs the predicted Q-Values for "Jump", "Run", or "Duck".



**Two Brilliant Innovations of DQN:**
1. **Experience Replay Buffer:** The agent doesn't learn from its actions immediately. It stores every memory (State, Action, Reward, Next State) in a massive database. During training, it grabs random batches of old memories to learn from. This stops the neural network from catastrophically forgetting old lessons.
2. **Target Network:** The Bellman equation requires the agent to guess the *future* Q-value to update its *current* Q-value. If both guesses come from the exact same network that is constantly changing, the math violently destabilizes. DQN uses a second, frozen "Target Network" to calculate the future, creating a stable target to aim at.

**Mathematical Foundation (DQN Loss Function):**
The network uses Mean Squared Error to minimize the difference between its current prediction and the Bellman target:
$$L(\theta) = \mathbb{E} \left[ \left( R + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta) \right)^2 \right]$$
*(Where $\theta$ are the weights of the main network, and $\theta^-$ are the frozen weights of the Target Network).*

**Example Problem:**
* **Complex Gaming:** DeepMind's famous breakthrough where an AI learned to play dozens of Atari 2600 games at a superhuman level just by looking at the raw pixels.

The visualization below demonstrates how a Neural Network approximates a continuous Q-Value function, bending a mathematical curve to learn the value of a continuous state space (like distance to a target), rather than using rigid spreadsheet cells.

In [ ]:
@interact(training_steps=IntSlider(min=0, max=100, step=10, value=0, description='Network Updates'))
def plot_dqn_approximation(training_steps):
    states = np.linspace(0, 10, 100)
    true_q_values = 10 * np.exp(-0.5 * ((states - 5) / 1.0)**2) 
    initial_guess = np.ones_like(states) * 2 
    
    progress = training_steps / 100.0
    network_prediction = initial_guess * (1 - progress) + true_q_values * progress
    
    plt.figure(figsize=(10, 5))
    plt.plot(states, true_q_values, 'g--', lw=3, alpha=0.5, label='True Environment Q-Values (Hidden)')
    plt.plot(states, network_prediction, 'r-', lw=3, label=f'DQN Neural Network Output (Step {training_steps})')
    plt.fill_between(states, network_prediction, true_q_values, color='red', alpha=0.1, label='DQN Loss (Error)')
    plt.title("DQN Concept: Using a Neural Network to approximate continuous Q-Values", fontsize=14, fontweight='bold')
    plt.xlabel("Continuous State Space (e.g., Distance to Target)")
    plt.ylabel("Predicted Expected Reward (Q-Value)")
    plt.legend()
    plt.ylim(0, 12)
    plt.show()

interactive(children=(IntSlider(value=0, description='Network Updates', step=10), Output()), _dom_classes=('wi…

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import gymnasium as gym
import random
from collections import deque

This code implements the two brilliant innovations of DQN:
1. **The Replay Buffer:** A memory bank (using Python's `deque`) that stores past experiences.
2. **The Target Network:** A secondary, frozen neural network that stabilizes the Bellman math.



The neural network takes the 4 physical numbers of the CartPole environment (Cart Position, Cart Velocity, Pole Angle, Pole Angular Velocity) and outputs 2 Q-Values (Push Left vs. Push Right).


![CartPole](https://tse2.mm.bing.net/th/id/OIP.NzCJqymlg3Vdmz08YVu3xAHaEZ?w=624&h=370&rs=1&pid=ImgDetMain&o=7&rm=3)


![Gymnasium](https://gymnasium.farama.org/_images/lunar_lander.gif)

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, 24)
        self.fc2 = nn.Linear(24, 24)
        self.fc3 = nn.Linear(24, action_size)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return self.fc3(x) 

env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

batch_size = 32
gamma = 0.95      
epsilon = 1.0     
epsilon_min = 0.01
epsilon_decay = 0.995
learning_rate = 0.001

policy_net = QNetwork(state_size, action_size)
target_net = QNetwork(state_size, action_size)
target_net.load_state_dict(policy_net.state_dict())
optimizer = optim.Adam(policy_net.parameters(), lr=learning_rate)
memory = deque(maxlen=2000)

print("Starting real DQN Training... This will take a moment.")
episodes = 200

for e in range(episodes):
    state, _ = env.reset()
    state = torch.FloatTensor(state).unsqueeze(0)
    total_reward = 0
    
    for time_step in range(500):
        if random.random() <= epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = policy_net(state).argmax().item()
                
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        reward = reward if not done else -10
        next_state = torch.FloatTensor(next_state).unsqueeze(0)
        
        memory.append((state, action, reward, next_state, done))
        state = next_state
        total_reward += 1
        
        if done:
            print(f"Episode: {e+1}/{episodes} | Score: {time_step} | Epsilon: {epsilon:.2f}")
            break
            
        if len(memory) > batch_size:
            minibatch = random.sample(memory, batch_size)
            
            states = torch.cat([m[0] for m in minibatch])
            actions = torch.LongTensor([m[1] for m in minibatch]).unsqueeze(1)
            rewards = torch.FloatTensor([m[2] for m in minibatch])
            next_states = torch.cat([m[3] for m in minibatch])
            dones = torch.FloatTensor([m[4] for m in minibatch])
            
            current_q_values = policy_net(states).gather(1, actions).squeeze(1)
            
            next_q_values = target_net(next_states).max(1)[0]
            expected_q_values = rewards + (gamma * next_q_values * (1 - dones))
            
            loss = F.mse_loss(current_q_values, expected_q_values.detach())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay
    if e % 10 == 0:
        target_net.load_state_dict(policy_net.state_dict())

env.close()
print("DQN Training Complete!")

Starting real DQN Training... This will take a moment.
Episode: 1/200 | Score: 17 | Epsilon: 1.00
Episode: 2/200 | Score: 10 | Epsilon: 0.99
Episode: 3/200 | Score: 27 | Epsilon: 0.99
Episode: 4/200 | Score: 26 | Epsilon: 0.99
Episode: 5/200 | Score: 28 | Epsilon: 0.98
Episode: 6/200 | Score: 16 | Epsilon: 0.98
Episode: 7/200 | Score: 13 | Epsilon: 0.97
Episode: 8/200 | Score: 21 | Epsilon: 0.97
Episode: 9/200 | Score: 51 | Epsilon: 0.96
Episode: 10/200 | Score: 25 | Epsilon: 0.96
Episode: 11/200 | Score: 48 | Epsilon: 0.95
Episode: 12/200 | Score: 21 | Epsilon: 0.95
Episode: 13/200 | Score: 19 | Epsilon: 0.94
Episode: 14/200 | Score: 41 | Epsilon: 0.94
Episode: 15/200 | Score: 13 | Epsilon: 0.93
Episode: 16/200 | Score: 84 | Epsilon: 0.93
Episode: 17/200 | Score: 16 | Epsilon: 0.92
Episode: 18/200 | Score: 18 | Epsilon: 0.92
Episode: 19/200 | Score: 42 | Epsilon: 0.91
Episode: 20/200 | Score: 16 | Epsilon: 0.91
Episode: 21/200 | Score: 19 | Epsilon: 0.90
Episode: 22/200 | Score: 83 | 

### 3. Policy Gradients (PPO / A3C)

Both Q-Learning and DQN are "Value-Based". They learn the *value* of every state, and the policy is just an afterthought ("pick the highest value"). 

**Policy Gradients** skip the middleman. They do not care about calculating exact Q-values. Instead, the Neural Network directly outputs the **Policy** itself: a probability distribution over the available actions!
Instead of outputting "Jumping is worth 42 points", the network outputs "There is an 80% chance you should Jump right now."



**Why is this better?**
1. **Continuous Actions:** Q-Learning can only output discrete buttons (Up, Down, Left, Right). Policy Gradients can output smooth, continuous numbers (e.g., "Turn the steering wheel exactly 14.2 degrees").
2. **Stochastic Policies:** In games like Rock-Paper-Scissors, if your AI is deterministic (always plays Rock), it will lose. Policy Gradients can learn to output a 33% / 33% / 33% probability, keeping its strategy unguessable!

**Mathematical Foundation (The Objective Function):**
We want to maximize the total expected reward $J(\theta)$. Using calculus, we update the network weights ($\theta$) by moving them in the direction that makes highly rewarded actions more probable:
$$\nabla J(\theta) = \mathbb{E} \left[ \nabla_\theta \log \pi_\theta(a|s) \cdot A(s, a) \right]$$
* $\pi_\theta(a|s)$: The Policy (The probability of taking action $a$ in state $s$).
* $A(s, a)$: The **Advantage**. Was taking this action *better* than the average expected outcome? If yes, increase the probability of this action. If no, decrease it!

*(Note: **PPO** (Proximal Policy Optimization) and **A3C** (Asynchronous Advantage Actor-Critic) are the modern, highly stable, industry-standard versions of Policy Gradients used by OpenAI).*

**Example Problem:**
* **Robotics & Autonomous Driving:** Training a Boston Dynamics robot dog to walk. The actions aren't "buttons"—they are exact torque voltages sent to 12 different motors simultaneously.

The interactive simulation below shows a continuous Policy Gradient learning to aim an artillery cannon. Watch how the "Policy" (a Gaussian probability distribution) shifts its mean angle and shrinks its variance to lock perfectly onto the target!

In [ ]:
@interact(epochs=IntSlider(min=0, max=100, step=10, value=0, description='Gradient Steps'))
def plot_policy_gradient(epochs):
    target_angle = 65.0 
    initial_mean = 30.0
    initial_std = 15.0 
    
    progress = epochs / 100.0

    current_mean = initial_mean + (target_angle - initial_mean) * progress
    current_std = initial_std - (initial_std - 2.0) * progress 
    angles = np.linspace(0, 90, 300)
    policy_distribution = stats.norm.pdf(angles, current_mean, current_std)
    
    plt.figure(figsize=(10, 5))
    plt.axvline(target_angle, color='green', linestyle='--', lw=3, label='Target Angle (Unknown to AI)')
    plt.plot(angles, policy_distribution, color='purple', lw=3, label=f'AI Action Policy (Mean: {current_mean:.1f}°)')
    plt.fill_between(angles, 0, policy_distribution, color='purple', alpha=0.2)
    plt.title(f"Continuous Policy Gradient (Step {epochs})\nDirectly optimizing the probability of taking the correct action!", fontsize=14, fontweight='bold')
    plt.xlabel("Action Space: Cannon Angle (Degrees)")
    plt.ylabel("Probability of taking action π(a|s)")
    plt.xlim(0, 90)
    plt.ylim(0, 0.25)
    plt.legend(loc='upper left')
    plt.show()

interactive(children=(IntSlider(value=0, description='Gradient Steps', step=10), Output()), _dom_classes=('wid…

Unlike DQN, which tries to calculate exact point values for actions, Policy Gradients output the literal probability of taking an action. 



This code implements the classic **REINFORCE** algorithm. The network plays an entire episode from start to finish, stores all the states and actions in a list, and then calculates the total reward. 
* If the total reward was good, it mathematically adjusts the network weights to make those specific actions more likely in the future. 
* If the reward was bad, it suppresses those actions.

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, 32)
        self.fc2 = nn.Linear(32, action_size)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        return F.softmax(self.fc2(x), dim=1) 

env = gym.make('CartPole-v1')
policy = PolicyNetwork(env.observation_space.shape[0], env.action_space.n)
optimizer = optim.Adam(policy.parameters(), lr=0.01)
gamma = 0.99

print("Starting real Policy Gradient (REINFORCE) Training...")
episodes = 300

for e in range(episodes):
    state, _ = env.reset()
    saved_log_probs = []
    rewards = []
    
    for time_step in range(500):
        state_tensor = torch.FloatTensor(state).unsqueeze(0)

        action_probs = policy(state_tensor)
        m = Categorical(action_probs)
        action = m.sample()
        
        saved_log_probs.append(m.log_prob(action))
        state, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(reward)
        
        if terminated or truncated:
            break
  
    returns = []
    R = 0
    for r in rewards[::-1]:
        R = r + gamma * R
        returns.insert(0, R)
        
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-9)
    
    policy_loss = []
    for log_prob, R in zip(saved_log_probs, returns):
        policy_loss.append(-log_prob * R)
        
    optimizer.zero_grad()
    policy_loss = torch.cat(policy_loss).sum()
    policy_loss.backward()
    optimizer.step()
    
    if e % 20 == 0:
        print(f"Episode {e} | Total Reward for this run: {sum(rewards):.0f}")

env.close()
print("Policy Gradient Training Complete!")

Starting real Policy Gradient (REINFORCE) Training...
Episode 0 | Total Reward for this run: 20
Episode 20 | Total Reward for this run: 42
Episode 40 | Total Reward for this run: 61
Episode 60 | Total Reward for this run: 150
Episode 80 | Total Reward for this run: 118
Episode 100 | Total Reward for this run: 92
Episode 120 | Total Reward for this run: 281
Episode 140 | Total Reward for this run: 488
Episode 160 | Total Reward for this run: 410
Episode 180 | Total Reward for this run: 143
Episode 200 | Total Reward for this run: 126
Episode 220 | Total Reward for this run: 134
Episode 240 | Total Reward for this run: 78
Episode 260 | Total Reward for this run: 69
Episode 280 | Total Reward for this run: 105
Policy Gradient Training Complete!
